# Fixed WIDTH and NBASIS for TGLF and GFTM on the `kbm_free` KBM databases, and which FILTER is justified

Fusion_PhD-qp5.37 (KBM paper), re-analysed under the rule NXGRID $\ge$ NBASIS + 8 (Fusion_PhD-qp5.40). Population: the GS2-KBMs of the four `kbm_free` Latin hypercubes (SPR-045 203, M1 140, R4 208,
R3 187; user classifier, $\gamma>0$, `growth_rate_tolerance` $<0.05$, species tied as GyroRun built the decks), pooled: 738.

- **TGLF** `TGLF/Runs/LATIN_HYPERCUBE/<case>/kbm_free/fx_<filter>_<w>_nb<N>`: SAT_RULE 2 presets, `FIND_WIDTH` F,
  NBASIS_MIN = NBASIS_MAX = $N$, NXGRID 32, `USE_MHD_RULE` F, `USE_BPER` T, `USE_BPAR` T, IBRANCH $-1$.
- **GFTM** (stock field solve, FIELD_GALERKIN F) `GFTM/Runs/LATIN_HYPERCUBE/<case>/kbm_free/galF[_f0p0]_<w>_nb<N>_nx<X>`:
  NXGRID 32 up to NBASIS 24, 40 at NBASIS 32, 65 at 48, 64 at 56 (rule NXGRID $\ge$ NBASIS + 8; GFTM resets $2\,$NXGRID$-1>129$ to 24, so NBASIS 64 cannot comply).
  FILTER 0.5 unless the leaf name says `f0p0`. The qp5.37 cells that break the rule (GFTM NBASIS 32 at NXGRID 32, NBASIS 64 at 65; TGLF NBASIS 32 at NXGRID 32,
  which TGLF cannot fix because it resets NXGRID 40 to 16) are loaded and flagged `rule_ok` False, used only in the superseded-cell comparison, never in Parts 0, A or B.
- 24 widths, `WIDTH_ORACLE_GRID` 0.2 to 4.0.

**Error metrics** per sample: $r_\gamma=\log_{10}(\gamma/\gamma_{\rm GS2})$, $r_\omega=\log_{10}(\omega/\omega_{\rm GS2})$,
$r_\phi$ and $r_{A_\parallel}$ the $\log_{10}$ ratio of pyro's 95% `Extent` of $|\phi|^2$ and $|A_\parallel|^2$. Over the pooled
population: RMS of $r$, with a bootstrap 95% CI, and median $|r|$. A **miss** is a sample with no unstable mode or with the dominant
mode in the electron direction; misses are excluded from the metrics and plotted as their own panel.
Dominant mode = mode 0 (IBRANCH $-1$ sorts by $\gamma$), checked against `argmax("mode")` on every leaf.

In [ ]:
import os
for threads in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[threads] = "1"
import json
import re
from multiprocessing import get_context
from pathlib import Path
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator, StrMethodFormatter
import pyrokinetics
from pyrokinetics import PyroHypercube
from pyrokinetics.diagnostics.extent import Extent
from general_analysis import mode_classification as mc

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__, pyrokinetics.__file__)

data_root = Path(os.path.expandvars(os.environ["GK_DATA_ROOT"]))     # WORK: the GS2 kbm_free cubes
run_root = Path(os.path.expandvars("$GYRO_DATA_OUTPUT"))             # scratch: the new leaves (copied to WORK after the runs)
plots_dir = ROOT / "Plots" / "TGLF_GFTM_KBM_PAPER" / "kbm_fixed_width_nbasis"
csv_dir = Path("csv")
plots_dir.mkdir(parents=True, exist_ok=True); csv_dir.mkdir(exist_ok=True)
cases = ["SPR-045", "M1", "R4", "R3"]
widths = [0.2, 0.2297, 0.2639, 0.3031, 0.3482, 0.4, 0.4595, 0.5279, 0.6063, 0.6965, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6,
          1.9218, 2.3083, 2.7726, 3.3302, 4.0]    # registry.WIDTH_ORACLE_GRID
width_tag = lambda width: f"w{width:.4f}".replace(".", "p")
tglf_nbasis = [2, 4, 6, 8, 12, 16, 24, 32]
gftm_cells = {2: 32, 4: 32, 8: 32, 12: 32, 16: 32, 24: 32, 32: 40, 48: 65, 56: 64}   # NBASIS: NXGRID, all with NXGRID >= NBASIS + 8
superseded_cells = [(32, 32), (64, 65)]   # qp5.37 GFTM cells that break the rule
tglf_filters = {"f0p0": 0.0, "f2p0": 2.0, "fm1p5": -1.5}
gftm_filters = {"": 0.5, "_f0p0": 0.0}
tglf_filter_tag = os.environ.get("TGLF_FILTER_TAG", "fm1p5")    # the Part 0 winner, used for Parts A and B
metrics = ["gamma", "omega", "phi", "apar"]
metric_label = {"gamma": r"$\gamma$", "omega": r"$\omega$", "phi": r"$\phi$ extent", "apar": r"$A_\parallel$ extent"}
n_cpu = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
rng = np.random.default_rng(0)

## GS2 reference and population
The user's classifier on each `kbm_free` cube (as `results/kbm_fixed_width_nbasis/runs/population.py`, which built the leaves). GS2's
`phi_squared`/`apar_squared` sit on a union $\theta$ grid NaN-padded per sample, so the extent is taken per sample on its finite points.

In [ ]:
def tied_species(sample):
    # GyroRun's links for the kbm_free cubes (not stored in pyroscan.json): ions share ion1's a/L_T, every a/L_n is the electron's
    species = gs2_scan.sample_pyro(sample).local_species
    ions = [name for name in species.names if name != "electron"]
    for ion in ions[1:]:
        species[ion].inverse_lt = species[ions[0]].inverse_lt
    for ion in ions:
        species[ion].inverse_ln = species["electron"].inverse_ln
    species.enforce_quasineutrality("electron")
    return {name: (float(species[name].temp.m), float(species[name].dens.m), float(species[name].inverse_lt.m)) for name in species.names}


def extent_rows(power):
    # pyro's 95% rule per sample on its own finite theta points; power: (sample, theta) DataArray
    if np.isfinite(power.values).all():
        return Extent._compute(power, 0.95)[0].values
    return np.array([float(Extent._compute(row.dropna("theta").expand_dims("sample"), 0.95)[0][0]) if np.isfinite(row.values).any() else np.nan
                     for row in power])


gs2 = {}
for case in cases:
    leaf = data_root / "GS2/Runs/LATIN_HYPERCUBE" / case / "kbm_free"
    gs2_scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    gs2_scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    gs2_output = gs2_scan.gk_output.data
    names = np.array([str(name) for name in gs2_output.sample_name.values])
    with get_context("fork").Pool(n_cpu) as pool:
        species = pool.map(tied_species, range(len(names)), chunksize=8)
    heat = gs2_output.heat.sum("field").pint.dequantify()
    particle = gs2_output.particle.sum("field").pint.dequantify()

    def chi(group):   # (Q_s - 1.5 T_s Gamma_s) / (n_s T_s a/L_Ts) summed over the group, as mode_classification.indicators
        numerator = denominator = 0.0
        for name in group:
            temp, dens, inverse_lt = np.array([sp[name] for sp in species]).T
            numerator = numerator + heat.sel(species=name).values - 1.5 * temp * particle.sel(species=name).values
            denominator = denominator + dens * temp * inverse_lt
        return numerator / denominator
    ions = [str(s) for s in gs2_output.species.values if s != "electron"]
    with np.errstate(divide="ignore", invalid="ignore"):
        labels = np.asarray(mc.classify(dict(T=gs2_output.tearing_parameter.values.ravel(),
                                             omega=mc.ION_DIRECTION * gs2_output.mode_frequency.pint.dequantify().values.ravel(),
                                             chi_ratio=chi(["electron"]) / chi(ions))))
    gamma = gs2_output.growth_rate.pint.dequantify().values.ravel()
    tolerance = gs2_output.growth_rate_tolerance.pint.dequantify().values.ravel()
    kbm = np.isfinite(gamma) & (gamma > 0) & (tolerance < 0.05) & (labels == "KBM")
    selected = gs2_output.set_index(sample="sample_name").sel(sample=names[kbm])
    gs2[case] = dict(names=names[kbm], gamma=gamma[kbm], omega=mc.ION_DIRECTION * selected.mode_frequency.pint.dequantify().values.ravel(),
                     phi=extent_rows(selected.phi_squared.pint.dequantify().squeeze("kx").transpose("sample", "theta")),
                     apar=extent_rows(selected.apar_squared.pint.dequantify().squeeze("kx").transpose("sample", "theta")),
                     units=re.sub(r"_pyroscan_base\d+", "", str(gs2_output.growth_rate.data.units)))
    print(case, "GS2-KBMs", kbm.sum(), "| median GS2 extent phi", np.median(gs2[case]["phi"]), "apar", np.median(gs2[case]["apar"]), flush=True)
    del gs2_output, selected
reference = pd.concat([pd.DataFrame(dict(case=case, sample=gs2[case]["names"], **{f"{m}_gs2": gs2[case][m] for m in metrics})) for case in cases])
assert len(reference) == 738, len(reference)

## TGLF and GFTM leaves
One `PyroHypercube` per leaf; settings from the leaf's `manifest.json` (`code_flags` GyroRun put on the GS2 source deck), checked
against the leaf name. Per sample: the dominant mode's $\gamma$, $\omega$ (ion direction positive) and extents, plus, from every mode,
the number of unstable modes and of those with an odd $\phi$ (`phi_even_fraction` $<0.5$, pyro `FieldLine.compute_linear_parity`,
stored by the save job's `--tearing`) and an even, tearing-parity $A_\parallel$ (`apar_even_fraction` $>0.5$).

In [ ]:
def leaves():
    for case in cases:
        for tag, value in tglf_filters.items():
            for nbasis in tglf_nbasis:
                for width in widths:
                    yield "TGLF", case, value, nbasis, 32, width, run_root / "TGLF/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"fx_{tag}_{width_tag(width)}_nb{nbasis}"
        for tag, value in gftm_filters.items():
            for nbasis, nxgrid in [*gftm_cells.items(), *superseded_cells, (32, 65)]:
                for width in widths:
                    yield "GFTM", case, value, nbasis, nxgrid, width, run_root / "GFTM/Runs/LATIN_HYPERCUBE" / case / "kbm_free" / f"galF{tag}_{width_tag(width)}_nb{nbasis}_nx{nxgrid}"


def load_leaf(spec):
    code, case, filter_value, nbasis, nxgrid, width, leaf = spec
    if not (leaf / "pyroscan.nc").is_file():
        return None
    scan = PyroHypercube(pyroscan_json=leaf / "pyroscan.json", load_base_pyro=True)
    scan.load_gk_output(netcdf_file=leaf / "pyroscan.nc")
    flags = json.loads((leaf / "manifest.json").read_text())["code_flags"]
    assert (float(flags["WIDTH"]), float(flags["FILTER"]), int(flags["NBASIS_MAX"]), int(flags["NXGRID"])) == (width, filter_value, nbasis, nxgrid), leaf
    assert not flags["FIND_WIDTH"] and (code == "GFTM" or (flags["SAT_RULE"] == 2 and not flags["USE_MHD_RULE"] and flags["USE_BPAR"])), leaf
    output = scan.gk_output.data
    assert re.sub(r"_pyroscan_base\d+", "", str(output.growth_rate.data.units)) == gs2[case]["units"], "normalisations differ"
    names = [str(name) for name in output.sample_name.values]
    output = output.set_index(sample="sample_name")
    growth = output.growth_rate.pint.dequantify().transpose("sample", "mode")
    assert (growth.fillna(-np.inf).argmax("mode") == 0).all(), leaf     # IBRANCH -1: mode 0 is the dominant mode
    frequency = mc.ION_DIRECTION * output.mode_frequency.pint.dequantify().transpose("sample", "mode")
    unstable = growth.fillna(0).values > 0
    row = dict(code=code, case=case, filter=filter_value, nbasis=nbasis, nxgrid=nxgrid, width=width, sample=names,
               failed=growth.isnull().all("mode").values, gamma=growth.isel(mode=0).fillna(0).values, omega=frequency.isel(mode=0).values, n_unstable=unstable.sum(1))  # mode0-guard-allow: IBRANCH -1, argmax==0 asserted above
    for field in ("phi", "apar"):
        power = (np.abs(output[field].isel(mode=0).pint.dequantify().squeeze("kx")) ** 2).transpose("sample", "theta")  # mode0-guard-allow: as above
        row[field] = np.where(row["gamma"] > 0, extent_rows(power), np.nan)
    if "phi_even_fraction" in output:
        row["n_unstable_phi_odd"] = (unstable & (output.phi_even_fraction.pint.dequantify().squeeze().transpose("sample", "mode").values < 0.5)).sum(1)
        row["n_unstable_apar_tearing"] = (unstable & (output.apar_even_fraction.pint.dequantify().squeeze().transpose("sample", "mode").values > 0.5)).sum(1)
    return pd.DataFrame(row)


with get_context("fork").Pool(n_cpu) as pool:
    runs = pd.concat([table for table in pool.imap_unordered(load_leaf, list(leaves()), chunksize=4) if table is not None], ignore_index=True)
runs = runs.merge(reference, on=["case", "sample"], validate="many_to_one")
print("runs with no output (all modes NaN), dropped:", runs.failed.sum(), "of", len(runs), flush=True)
if runs.failed.any():
    print(runs[runs.failed].groupby(["code", "case", "filter", "nbasis", "nxgrid"]).size().to_string(), flush=True)
runs = runs[~runs.failed]
runs["rule_ok"] = runs.nxgrid >= runs.nbasis + 8
runs["miss"] = ~((runs.gamma > 0) & (runs.omega > 0))
for metric in metrics:
    runs[f"r_{metric}"] = np.where(runs.miss, np.nan, np.log10(runs[metric] / runs[f"{metric}_gs2"]))
print(runs.groupby(["code", "filter", "nxgrid", "nbasis"]).agg(leaves=("width", "nunique"), rows=("sample", "size")).to_string(), flush=True)


def summarise(group):
    # RMS of r over non-misses, its bootstrap 95% CI, median |r|, miss fraction
    out = dict(n=len(group), miss_fraction=group.miss.mean())
    for metric in metrics:
        values = group[f"r_{metric}"].dropna().values
        boot = np.sqrt((rng.choice(values, (1000, values.size)) ** 2).mean(1)) if values.size else np.full(1, np.nan)
        out.update({f"{metric}_rms": np.sqrt((values ** 2).mean()) if values.size else np.nan, f"{metric}_lo": np.percentile(boot, 2.5),
                    f"{metric}_hi": np.percentile(boot, 97.5), f"{metric}_median_abs": np.median(np.abs(values)) if values.size else np.nan})
    return pd.Series(out)


keys = ["code", "filter", "nbasis", "nxgrid", "width"]
argmin_width = lambda table, column: table.sort_values("width").pipe(lambda d: d.width.values[np.nanargmin(d[column].values)] if np.isfinite(d[column].values).any() else np.nan)
pooled = runs.groupby(keys).apply(summarise).reset_index()
per_case = runs.groupby(["case", *keys]).apply(summarise).reset_index()

## Part 0: is a FILTER justified on KBMs?
A filter is justified if it flattens the extent-vs-WIDTH curve toward 1 (mode selection stops depending on the width) and lowers the
$\gamma$ error away from the optimum, without adding misses. FILTER 0 is the reference arm. "Changed the dominant mode" compares each
sample's dominant $(\gamma,\omega)$ with the same sample, width and NBASIS at FILTER 0. TGLF $-0.1$ (drops every odd mode) is excluded
(user, 2026-11-06: "that is essentially tell TGLF what mode there will be").

In [ ]:
filter_rows = runs.merge(runs[runs["filter"] == 0][["code", "case", "sample", "nbasis", "nxgrid", "width", "gamma", "omega"]],
                         on=["code", "case", "sample", "nbasis", "nxgrid", "width"], suffixes=("", "_f0"))
filter_rows["changed"] = ~np.isclose(filter_rows.gamma, filter_rows.gamma_f0, rtol=1e-6) | ~np.isclose(filter_rows.omega, filter_rows.omega_f0, rtol=1e-6, equal_nan=True)
for field in ("phi", "apar"):
    filter_rows[f"{field}_log_ratio"] = np.log10(filter_rows[field] / filter_rows[f"{field}_gs2"])
quartile = lambda q: (lambda x: np.nanpercentile(x, q) if np.isfinite(x).any() else np.nan)
filter_curves = filter_rows.groupby(keys).agg(**{f"{field}_{stat}": (f"{field}_log_ratio", fn) for field in ("phi", "apar")
                                                 for stat, fn in (("median", "median"), ("q25", quartile(25)), ("q75", quartile(75)))},
                                             changed=("changed", "mean"), miss=("miss", "mean"),
                                             gamma_rms=("r_gamma", lambda r: np.sqrt(np.nanmean(r ** 2)))).reset_index()   # paired with FILTER 0 by the merge
part0_nbasis = {"TGLF": [4, 16], "GFTM": [16, 56]}   # TGLF NBASIS 32 is not rule-compliant
panels = [("phi", "log(Extent Ratio)", r"$\phi$"), ("apar", "log(Extent Ratio)", r"$A_\parallel$"),
          ("gamma_rms", r"RMS $\log_{10}(\gamma/\gamma_{\rm GS2})$", r"$\gamma$"), ("changed", "fraction", "dominant mode changed"),
          ("miss", "fraction", "misses")]
for code in part0_nbasis:
    fig, axes = plt.subplots(len(part0_nbasis[code]), len(panels), figsize=(26, 4.6 * len(part0_nbasis[code])), sharex=True, squeeze=False, constrained_layout=True)
    for row, nbasis in enumerate(part0_nbasis[code]):
        for column, (quantity, ylabel, title) in enumerate(panels):
            ax = axes[row, column]
            for colour, (filter_value, curve) in zip(("C0", "C3", "C2"), filter_curves[(filter_curves.code == code) & (filter_curves.nbasis == nbasis)
                                                                                         & (filter_curves.nxgrid == (gftm_cells[nbasis] if code == "GFTM" else 32))].groupby("filter")):
                if quantity in ("phi", "apar"):
                    ax.plot(curve.width, curve[f"{quantity}_median"], color=colour, marker="o", ms=3, label=f"FILTER={filter_value:g}")
                    ax.fill_between(curve.width, curve[f"{quantity}_q25"], curve[f"{quantity}_q75"], color=colour, alpha=0.15)
                else:
                    ax.plot(curve.width, curve[quantity], color=colour, marker="o", ms=3, label=f"FILTER={filter_value:g}")
            if quantity in ("phi", "apar"):
                ax.axhline(0, color="0.3", lw=1, marker="")
            ax.set_xscale("log")
            ax.set_title(f"{code} NBASIS {nbasis}, {title}", fontsize=16)
            ax.set_ylabel(ylabel, fontsize=16)
            ax.tick_params(axis="both", labelsize=14)
    for ax in axes[-1]:
        ax.set_xlabel("WIDTH", fontsize=16)
    axes[0, 0].legend(fontsize=16)
    lims = axes[0, 0].get_xlim()
    axes[0, 0].set_xticks([0.2, 0.5, 1, 2, 4])
    axes[0, 0].set_xlim(lims)
    axes[0, 0].xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    for ax in axes.flat:
        ax.xaxis.set_minor_locator(NullLocator())   # keep the log-y minor labels: most panels span under a decade
    fig.savefig(plots_dir / f"part0_filter_vs_width_{code}.png", dpi=150); fig.savefig(plots_dir / f"part0_filter_vs_width_{code}.pdf")

verdict = []
for (code, nbasis, filter_value), curve in filter_curves.groupby(["code", "nbasis", "filter"]):
    if nbasis not in part0_nbasis[code]:
        continue
    optimum = curve.width.values[np.nanargmin(curve.gamma_rms.values)]
    away = np.abs(np.log(curve.width.values / optimum)) > np.log(1.5)    # widths more than x1.5 from the gamma optimum
    verdict.append(dict(code=code, nbasis=nbasis, filter=filter_value,
                        phi_log_ratio_spread=np.ptp(curve.phi_median.dropna()), phi_log_ratio_mean_abs=curve.phi_median.abs().mean(),
                        apar_log_ratio_spread=np.ptp(curve.apar_median.dropna()), apar_log_ratio_mean_abs=curve.apar_median.abs().mean(),
                        gamma_rms_at_optimum=np.nanmin(curve.gamma_rms), gamma_rms_away=curve.gamma_rms[away].mean(),
                        miss_mean=curve.miss.mean(), changed_mean=curve.changed.mean()))
verdict = pd.DataFrame(verdict)
print(verdict.round(3).to_string())
verdict.to_csv(csv_dir / "part0_filter_verdict.csv", index=False)

# what each filter removes: unstable modes per sample at FILTER 0, and how many have odd phi / tearing-parity apar
removed = runs[(runs["filter"] == 0) & runs.n_unstable_phi_odd.notna()].groupby(["code", "case", "nbasis"]).agg(
    unstable_per_sample=("n_unstable", "mean"), phi_odd_per_sample=("n_unstable_phi_odd", "mean"),
    apar_tearing_per_sample=("n_unstable_apar_tearing", "mean"), samples_with_phi_odd_mode=("n_unstable_phi_odd", lambda x: (x > 0).mean())).reset_index()
gftm_removed = runs[runs.code == "GFTM"].pivot_table(index=["case", "nbasis", "width", "sample"], columns="filter", values="n_unstable").dropna()
removed = removed.merge((gftm_removed[0.0] - gftm_removed[0.5]).groupby(["case", "nbasis"]).mean().rename("gftm_f05_removed_per_sample").reset_index(),
                        on=["case", "nbasis"], how="left")
print(removed.round(3).to_string())
removed.to_csv(csv_dir / "part0_modes_removed.csv", index=False)

## Part A: the U-shapes, error against WIDTH per NBASIS
Arms: TGLF at the Part 0 FILTER, GFTM at FILTER 0.5 (stock), only cells with NXGRID $\ge$ NBASIS + 8 (NXGRID 32 to NBASIS 24, 40 at 32, 65 at 48, 64 at 56). Marked: the argmin of each metric at
the top NBASIS (filled) and at NBASIS 4 (open). Candidate widths: those argmins at the top NBASIS plus the low-NBASIS $\gamma$ argmin,
merged when within one grid step.

In [ ]:
arm_filter = {"TGLF": tglf_filters[tglf_filter_tag], "GFTM": 0.5}
arm = pooled[[(row.filter == arm_filter[row.code]) and row.nxgrid >= row.nbasis + 8 and (row.code == "TGLF" or (row.nbasis, row.nxgrid) in gftm_cells.items()) for row in pooled.itertuples()]]
arm_filter = {code: value for code, value in arm_filter.items() if (arm.code == code).any()}   # a code whose leaves are not run yet is skipped
top_nbasis = {code: arm[arm.code == code].nbasis.max() for code in arm_filter}
candidates = {}
for code in arm_filter:
    data = arm[arm.code == code]
    fig, axes = plt.subplots(1, 5, figsize=(30, 6), sharex=True, constrained_layout=True)
    colours = plt.cm.viridis(np.linspace(0, 0.95, data.nbasis.nunique()))
    for colour, (nbasis, curve) in zip(colours, data.groupby("nbasis")):
        for ax, metric in zip(axes, metrics):
            ax.plot(curve.width, curve[f"{metric}_rms"], "-", color=colour, label=f"NBASIS {nbasis}")
            ax.fill_between(curve.width, curve[f"{metric}_lo"], curve[f"{metric}_hi"], color=colour, alpha=0.15)
            if nbasis in (top_nbasis[code], 4):
                best = curve.iloc[np.nanargmin(curve[f"{metric}_rms"].values)]
                ax.plot(best.width, best[f"{metric}_rms"], "o", ms=10, color=colour, mfc=colour if nbasis == top_nbasis[code] else "none", mew=2)
        axes[4].plot(curve.width, curve.miss_fraction, "-", color=colour)
    for ax, metric in zip(axes, metrics):
        ax.set_title(f"{code}, {metric_label[metric]}", fontsize=16); ax.set_yscale("log")
        ax.set_ylabel(f"RMS $\\log_{{10}}$ ratio to GS2", fontsize=16)
    axes[4].set_title(f"{code}, misses", fontsize=16); axes[4].set_ylabel("fraction", fontsize=16)
    for ax in axes:
        ax.set_xscale("log"); ax.set_xlabel("WIDTH", fontsize=16); ax.tick_params(axis="both", labelsize=14)
    lims = axes[0].get_xlim()
    axes[0].set_xticks([0.2, 0.5, 1, 2, 4]); axes[0].set_xlim(lims)
    axes[0].xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    for ax in axes:
        ax.xaxis.set_minor_locator(NullLocator())   # keep the log-y minor labels: most panels span under a decade
    axes[0].legend(fontsize=12, ncol=2)
    fig.savefig(plots_dir / f"partA_error_vs_width_{code}.png", dpi=150); fig.savefig(plots_dir / f"partA_error_vs_width_{code}.pdf")

    argmins = {metric: argmin_width(data[data.nbasis == top_nbasis[code]], f"{metric}_rms") for metric in metrics}
    argmins["gamma_nb4"] = argmin_width(data[data.nbasis == 4], "gamma_rms")
    merged = []
    for width in sorted({width for width in argmins.values() if np.isfinite(width)}):
        if not merged or widths.index(width) - widths.index(merged[-1][-1]) > 1:
            merged.append([width])
        else:
            merged[-1].append(width)
    candidates[code] = [group[len(group) // 2] for group in merged]
    per_case_argmin = per_case[(per_case.code == code) & (per_case.nbasis == top_nbasis[code]) & (per_case["filter"] == arm_filter[code])
                               & (per_case.nxgrid == (32 if code == "TGLF" else gftm_cells[top_nbasis[code]]))].groupby("case").apply(
        lambda d: pd.Series({metric: argmin_width(d, f"{metric}_rms") for metric in metrics}))
    print(code, "per-case argmin at the top NBASIS:\n", per_case_argmin.to_string())
    if len(candidates[code]) < 3:   # pooled argmins coincide: bracket them with the per-case gamma argmins' extremes
        for width in (per_case_argmin.gamma.min(), per_case_argmin.gamma.max()):
            if min(abs(widths.index(width) - widths.index(c)) for c in candidates[code]) > 1:
                candidates[code].append(width)
        candidates[code].sort()
    print(code, "argmins:", argmins, "-> candidates", candidates[code])
candidate_table = pd.concat([arm[(arm.code == code) & arm.width.isin(candidates[code])] for code in arm_filter])
candidate_table.to_csv(csv_dir / "candidate_widths_by_nbasis.csv", index=False)

## GFTM NXGRID check: NBASIS 32 at NXGRID 32 (superseded), 40 (rule) and 65 (50-sample subset)
On a 50-sample subset (seed 0) NBASIS 32 ran at NXGRID 65 as well as 32 and 40: does the NXGRID change move the result?

In [ ]:
subset = runs[(runs.code == "GFTM") & (runs["filter"] == 0.5) & (runs.nbasis == 32)]
subset = subset[subset.set_index(["case", "sample"]).index.isin(subset[subset.nxgrid == 65].set_index(["case", "sample"]).index)]
nxgrid_check = subset.groupby(["nxgrid", "width"]).apply(summarise).reset_index()
print(nxgrid_check.pivot(index="width", columns="nxgrid", values=["gamma_rms", "gamma_lo", "gamma_hi", "phi_rms"]).round(3).to_string())
paired = subset.pivot_table(index=["case", "sample", "width"], columns="nxgrid", values="gamma")
for low, high in [(32, 40), (40, 65), (32, 65)]:
    print(f"median |log10 gamma(nx{high})/gamma(nx{low})|:", np.nanmedian(np.abs(np.log10(paired[high] / paired[low]))), "over", len(paired), "pairs")
nxgrid_check.to_csv(csv_dir / "gftm_nb32_nxgrid_check.csv", index=False)

## Part B: convergence in NBASIS at the candidate widths, and the cost
Wall time per run from the dedicated benchmark (`timing.csv`: one exclusive node, one core and thread per run, 30 pooled GS2-KBM samples,
3 repeats, at one width per code), median and IQR per NBASIS, with a fit $t\propto N^p$. **Converged NBASIS** at a width: the smallest
$N$ from which every larger NBASIS keeps every metric's RMS inside the top-$N$ bootstrap 95% CI. **Recommended width:** the candidate reaching that with
the fewest $N$.

In [ ]:
timing_file = Path(os.environ.get("TIMING_CSV", "timing.csv"))
timing = pd.read_csv(timing_file) if timing_file.is_file() else None
timing_nx8_file = Path(os.environ.get("TIMING_NX8_CSV", "timing_nx8.csv"))   # qp5.40: the new (NBASIS, NXGRID) pairs, same protocol
if timing is not None and timing_nx8_file.is_file():
    timing = pd.concat([timing, pd.read_csv(timing_nx8_file)])
if timing is not None:
    timing = timing[(timing.nxgrid >= timing.nbasis + 8)]   # compliant cells only (TGLF at NXGRID 32 up to NBASIS 24)
# Part B also draws the widths around GFTM's U bottom (user, 2026-11-11: "a few more widths"); the recommendation rule is unchanged.
partB_widths = {code: sorted(set(candidates[code]) | ({0.3482, 0.4595, 0.5279, 0.6063, 0.8, 1.1} if code == "GFTM" else set())) for code in arm_filter}
recommendation = []
for code in arm_filter:
    data = arm[arm.code == code]
    fig, axes = plt.subplots(1, 6, figsize=(36, 6), sharex=True, constrained_layout=True)
    for width in partB_widths[code]:
        curve = data[data.width == width].sort_values("nbasis")
        top = curve[curve.nbasis == top_nbasis[code]].iloc[0]
        for ax, metric in zip(axes, metrics):
            ax.errorbar(curve.nbasis, curve[f"{metric}_rms"], yerr=[curve[f"{metric}_rms"] - curve[f"{metric}_lo"], curve[f"{metric}_hi"] - curve[f"{metric}_rms"]],
                        fmt="o-", capsize=3, label=f"WIDTH {width:g}")
        axes[4].plot(curve.nbasis, curve.miss_fraction, "o-")
        # inside the top-N bootstrap CI for every metric, at this N and every larger one (an RMS BELOW the CI is not convergence)
        inside = np.all([(curve[f"{metric}_rms"] >= top[f"{metric}_lo"]) & (curve[f"{metric}_rms"] <= top[f"{metric}_hi"]) for metric in metrics], axis=0)
        stays = np.cumprod(inside[::-1])[::-1].astype(bool)
        nbasis_converged = int(curve.nbasis.values[stays].min())
        at = curve[curve.nbasis == nbasis_converged]
        recommendation.append(dict(code=code, width=width, nbasis_converged=nbasis_converged,
                                   **{f"{metric}_rms_at_converged": at[f"{metric}_rms"].iloc[0] if len(at) else np.nan for metric in metrics},
                                   **{f"{metric}_rms_top": top[f"{metric}_rms"] for metric in metrics}, miss_at_converged=at.miss_fraction.iloc[0] if len(at) else np.nan))
    if timing is not None and (timing.code == code).any():
        times = timing[timing.code == code].groupby("nbasis").seconds.describe()
        axes[5].errorbar(times.index, times["50%"], yerr=[times["50%"] - times["25%"], times["75%"] - times["50%"]], fmt="o-", capsize=3)
        exponent, _ = np.polyfit(np.log(times.index[times.index >= 8]), np.log(times["50%"][times.index >= 8]), 1)
        axes[5].set_title(f"{code}, cost", fontsize=16)   # the fitted exponent goes in the README
        print(code, "timing\n", times[["count", "25%", "50%", "75%"]].round(3).to_string(), "\n exponent", round(exponent, 2))
        times.to_csv(csv_dir / f"timing_{code}.csv")
        axes[5].set_yscale("log")
    for ax, metric in zip(axes, metrics):
        ax.set_title(f"{code}, {metric_label[metric]}", fontsize=16); ax.set_yscale("log")
        ax.set_ylabel(f"RMS $\\log_{{10}}$ ratio to GS2", fontsize=16)
    axes[4].set_title(f"{code}, misses", fontsize=16); axes[4].set_ylabel("fraction", fontsize=16)
    axes[5].set_ylabel("wall time per run [s]", fontsize=16)
    for ax in axes:
        ax.set_xscale("log", base=2); ax.set_xlabel("NBASIS", fontsize=16); ax.tick_params(axis="both", labelsize=14)
        ax.set_xticks(sorted(data.nbasis.unique())); ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}")); ax.xaxis.set_minor_locator(NullLocator())   # keep the log-y minor labels: most panels span under a decade
    axes[0].legend(fontsize=12)
    fig.savefig(plots_dir / f"partB_error_vs_nbasis_{code}.png", dpi=150); fig.savefig(plots_dir / f"partB_error_vs_nbasis_{code}.pdf")
recommendation = pd.DataFrame(recommendation)
print(recommendation.round(3).to_string())
recommendation.to_csv(csv_dir / "recommendation.csv", index=False)
per_case[per_case.width.isin([w for c in candidates.values() for w in c])].to_csv(csv_dir / "per_case_candidates.csv", index=False)
pooled.to_csv(csv_dir / "pooled_summary.csv", index=False)